# 01 — Acquire and Prepare Seattle Data

This script downloads Seattle ArcGIS services, ceates a project file geodatabase aside from the default, and converts the service layers to local feature classes. 

The primary inventory is from the Seattle Paks and Recreation's public tree layer. Optional exposure and equity layers were added by pasting their REST layer URLs into their respective place holders 'DATASETS'.

> **Environment:** Run these notebooks inside an ArcGIS Pro coloned Python environment with `arcpy`, `pandas`, `numpy`, `matplotlib`, and `requests`, and the ArcGIS API for Python installed.

This workflow is a **portfolio sumulation of municipal asset management**. It prioritizes trees for inspection; it does **not** predict failure or replace an ISA-certified arborist's assessment. 

In [ ]:
from pathlib import Path
import os
from urllib.parse import urlencode
import json
import requests
import arcpy

arcpy.env.overwriteOutput = True

#--------------------------------------------------
#Configuration -- change PROJECT_ROOT to your own project folder.
#--------------------------------------------------

PROJECT_ROOT = Path(os.environ.get("SEATTLE_TREE_PROJECT_ROOT", Path.cwd())).resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
GDB = PROJECT_ROOT / "data" / "Seattle_Tree_Assets.gdb"

RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GDB.parent.mkdir(parents=True, exist_ok=True)

if not arcpy.Exists(str(GDB)):
    arcpy.management.CreateFileGDB(str(GDB.parent), GDB.name)

# Seattle public REST layers confirmed when this template was prepared.
# Add a URL for schools, major roads, and RSE polygons once selected. 
DATASETS = {
    "spr_trees": {
        "url": "https://services.arcgis.com/ZOyb2t4B0UYuYNYH/arcgis/rest/services/SPR_Tree_View/FeatureServer/0",
        "where": "STATUS = 'Existing'",
        "required": True,
    },
    "sidewalks": {
        "url": "https://services.arcgis.com/ZOyb2t4B0UYuYNYH/arcgis/rest/services/Sidewalks_CDL/FeatureServer/0",
        "where": "1=1",
        "required": True,
    },
    "trails": {
        "url": "https://services.arcgis.com/ZOyb2t4B0UYuYNYH/arcgis/rest/services/SPR_Trails/FeatureServer/0",
        "where": "1=1",
        "required": False,
    },
    # Paste ArcGIS REST FeatureServer layer URLs below.
    "schools": {"url": "https://services.arcgis.com/ZOyb2t4B0UYuYNYH/arcgis/rest/services/Seattle_Schools/FeatureServer/8", 
                "where": "1=1", 
                "required": False},
    "streets": { "url": (
        "https://services.arcgis.com/ZOyb2t4B0UYuYNYH/"
        "arcgis/rest/services/Seattle_Streets_1/FeatureServer/0"), "where": "1=1","required": True,},
    
    "rse_index": {"url": "https://services.arcgis.com/ZOyb2t4B0UYuYNYH/ArcGIS/rest/services/Race_and_Social_Equity_Composite_Index_Current/FeatureServer/0", 
                  "where": "1=1", 
                  "required": False},
}

TARGET_SR = arcpy.SpatialReference(2926)  #NAD83(HARN) / Washington North (ftUS)
print(f"Project geodatabase: {GDB}")


In [ ]:
temp_fc = str(GDB / "tmp_spr_trees")
output_fc = str(GDB / "spr_trees")

for path in [temp_fc, output_fc]:
    if arcpy.Exists(path):
        arcpy.management.Delete(path)
        print(f"Deleted: {path}")

geojson_file = RAW_DIR / "spr_trees.geojson"

if geojson_file.exists():
    geojson_file.unlink()
    print("Deleted old projected-coordinate GeoJSON.")

def service_to_geojson(
    layer_url: str,
    output_file: Path,
    where: str = "1=1",
    batch_size: int = 500,
) -> Path:
    """
    Download all records from an ArcGIS REST feature layer as GeoJSON.

    Uses POST requests to avoid excessively long URLs when querying
    large batches of object IDs.
    """
    query_url = f"{layer_url.rstrip('/')}/query"

    # ---------------------------------------------------------------
    # 1. Request the matching object IDs
    # ---------------------------------------------------------------
    id_params = {
        "where": where,
        "returnIdsOnly": "true",
        "f": "json",
    }

    id_response = requests.post(
        query_url,
        data=id_params,
        timeout=120,
    )
    id_response.raise_for_status()

    id_json = id_response.json()

    if "error" in id_json:
        raise RuntimeError(
            f"ArcGIS service error while retrieving IDs: {id_json['error']}"
        )

    object_ids = sorted(id_json.get("objectIds") or [])

    if not object_ids:
        raise RuntimeError(
            f"No records returned from {layer_url}. "
            f"WHERE clause: {where}. Response: {id_json}"
        )

    print(f"Found {len(object_ids):,} matching records.")

    # ---------------------------------------------------------------
    # 2. Download features in batches
    # ---------------------------------------------------------------
    features = []
    crs = None

    for start in range(0, len(object_ids), batch_size):
        batch = object_ids[start:start + batch_size]

        query_params = {
            "objectIds": ",".join(map(str, batch)),
            "outFields": "*",
            "returnGeometry": "true",
            "outSR": "4326",
            "f": "geojson",
        }

        response = requests.post(
            query_url,
            data=query_params,
            timeout=180,
        )
        response.raise_for_status()

        payload = response.json()

        if "error" in payload:
            raise RuntimeError(
                f"ArcGIS service error for records "
                f"{start + 1}–{start + len(batch)}: "
                f"{payload['error']}"
            )

        batch_features = payload.get("features", [])

        if not batch_features:
            print(
                f"Warning: batch {start + 1}–"
                f"{start + len(batch)} returned no features."
            )

        features.extend(batch_features)

        if crs is None:
            crs = payload.get("crs")

        print(
            f"Downloaded "
            f"{min(start + batch_size, len(object_ids)):,}"
            f"/{len(object_ids):,}"
        )

    # ---------------------------------------------------------------
    # 3. Save combined GeoJSON
    # ---------------------------------------------------------------
    collection = {
        "type": "FeatureCollection",
        "features": features,
    }

    if crs:
        collection["crs"] = crs

    output_file.write_text(
        json.dumps(collection),
        encoding="utf-8",
    )

    print(f"Saved {len(features):,} features to:")
    print(output_file)

    return output_file


In [ ]:
def geojson_to_feature_class(
    geojson_file: Path,
    output_fc: str,
) -> str:
    """
    Convert a GeoJSON FeatureCollection into a projected geodatabase
    feature class.

    Detects the GeoJSON geometry type and supplies the corresponding
    ArcGIS JSONToFeatures geometry_type parameter.
    """

    import json

    temp_fc = str(
        GDB / f"tmp_{Path(output_fc).name}"
    )

    # ---------------------------------------------------------------
    # Remove outputs from previous attempts
    # ---------------------------------------------------------------
    for dataset in [temp_fc, output_fc]:
        if arcpy.Exists(dataset):
            arcpy.management.Delete(dataset)

    print("\nConverting GeoJSON:")
    print(geojson_file)

    # ---------------------------------------------------------------
    # Read the GeoJSON and find the first usable geometry
    # ---------------------------------------------------------------
    with open(
        geojson_file,
        "r",
        encoding="utf-8",
    ) as file:
        geojson = json.load(file)

    features = geojson.get("features", [])

    if not features:
        raise RuntimeError(
            f"No features found in {geojson_file}"
        )

    first_feature = next(
        (
            feature
            for feature in features
            if feature.get("geometry")
            and feature["geometry"].get("coordinates") is not None
        ),
        None,
    )

    if first_feature is None:
        raise RuntimeError(
            f"No usable geometries found in {geojson_file}"
        )

    geojson_geometry = first_feature["geometry"]["type"]

    # GeoJSON names do not exactly match ArcGIS tool parameters.
    geometry_lookup = {
        "Point": "POINT",
        "MultiPoint": "MULTIPOINT",
        "LineString": "POLYLINE",
        "MultiLineString": "POLYLINE",
        "Polygon": "POLYGON",
        "MultiPolygon": "POLYGON",
    }

    arcgis_geometry = geometry_lookup.get(
        geojson_geometry
    )

    if arcgis_geometry is None:
        raise ValueError(
            f"Unsupported GeoJSON geometry type: "
            f"{geojson_geometry}"
        )

    print(
        f"GeoJSON geometry: {geojson_geometry}"
    )
    print(
        f"ArcGIS geometry parameter: {arcgis_geometry}"
    )

    # ---------------------------------------------------------------
    # Convert GeoJSON to a temporary feature class
    # ---------------------------------------------------------------
    arcpy.conversion.JSONToFeatures(
        in_json_file=str(geojson_file),
        out_features=temp_fc,
        geometry_type=arcgis_geometry,
    )

    temp_count = int(
        arcpy.management.GetCount(temp_fc)[0]
    )

    print(
        f"Temporary feature count: {temp_count:,}"
    )

    if temp_count == 0:
        raise RuntimeError(
            f"JSONToFeatures created zero records for "
            f"{geojson_file}. Detected geometry type was "
            f"{geojson_geometry}."
        )

    # ---------------------------------------------------------------
    # Check the input coordinate system
    # ---------------------------------------------------------------
    source_sr = (
        arcpy.Describe(temp_fc)
        .spatialReference
    )

    print("Reported input coordinate system:")
    print("  Name:", source_sr.name)
    print("  WKID:", source_sr.factoryCode)

    # GeoJSON coordinates shown in your test are valid longitude/
    # latitude coordinates around Seattle.
    wgs84 = arcpy.SpatialReference(4326)

    if (
        source_sr.name in ("Unknown", "")
        or source_sr.factoryCode == 0
    ):
        print(
            "Defining temporary feature class as WGS 84."
        )

        arcpy.management.DefineProjection(
            in_dataset=temp_fc,
            coor_system=wgs84,
        )

        source_sr = (
            arcpy.Describe(temp_fc)
            .spatialReference
        )

    print("Actual input coordinate system:")
    print("  Name:", source_sr.name)
    print("  WKID:", source_sr.factoryCode)

    # ---------------------------------------------------------------
    # Project to Washington State Plane North
    # ---------------------------------------------------------------
    if (
        source_sr.factoryCode
        == TARGET_SR.factoryCode
    ):
        print(
            "Input is already in the target coordinate system."
        )

        arcpy.management.CopyFeatures(
            in_features=temp_fc,
            out_feature_class=output_fc,
        )

    else:
        transformations = arcpy.ListTransformations(
            source_sr,
            TARGET_SR,
        )

        transformation = (
            transformations[0]
            if transformations
            else None
        )

        print("Projecting to:")
        print("  Name:", TARGET_SR.name)
        print("  WKID:", TARGET_SR.factoryCode)

        if transformation:
            print(
                "Transformation:",
                transformation,
            )

            arcpy.management.Project(
                in_dataset=temp_fc,
                out_dataset=output_fc,
                out_coor_system=TARGET_SR,
                transform_method=transformation,
            )

        else:
            arcpy.management.Project(
                in_dataset=temp_fc,
                out_dataset=output_fc,
                out_coor_system=TARGET_SR,
            )

    # ---------------------------------------------------------------
    # Validate and clean up
    # ---------------------------------------------------------------
    final_count = int(
        arcpy.management.GetCount(output_fc)[0]
    )

    final_sr = (
        arcpy.Describe(output_fc)
        .spatialReference
    )

    print(
        f"Final feature count: {final_count:,}"
    )
    print(
        "Final coordinate system:",
        final_sr.name,
    )
    print(
        "Output:",
        output_fc,
    )

    if final_count == 0:
        raise RuntimeError(
            f"The final feature class is empty: "
            f"{output_fc}"
        )

    arcpy.management.Delete(temp_fc)

    return output_fc

In [ ]:
failed_fc = str(GDB / "spr_trees")

if arcpy.Exists(failed_fc):
    arcpy.management.Delete(failed_fc)
    print("Deleted incomplete feature class.")

download_log = []

for name, settings in DATASETS.items():
    url = settings["url"].strip()
    if not url:
        message = f"{name}: skipped — no service URL configured."
        print(message)
        download_log.append({"dataset": name, "status": "skipped", "detail": message})
        continue

    try:
        print(f"\nProcessing {name}...")
        geojson_path = RAW_DIR / f"{name}.geojson"
        output_fc = str(GDB / name)
        service_to_geojson(url, geojson_path, settings["where"])
        geojson_to_feature_class(geojson_path, output_fc)
        count = int(arcpy.management.GetCount(output_fc)[0])
        print(f"{name}: {count:,} records saved to {output_fc}")
        download_log.append({"dataset": name, "status": "success", "records": count})
    except Exception as exc:
        download_log.append({"dataset": name, "status": "failed", "detail": str(exc)})
        if settings["required"]:
            raise
        print(f"Optional dataset {name} failed: {exc}")

with open(OUTPUT_DIR / "download_log.json", "w", encoding="utf-8") as file:
    json.dump(download_log, file, indent=2)

In [ ]:
# Inspect the imported tree schema before continuing.
tree_fc = str(GDB / "spr_trees")
fields = [(f.name, f.aliasName, f.type) for f in arcpy.ListFields(tree_fc)]
for row in fields:
    print(row)

print("\nTree extent:", arcpy.Describe(tree_fc).extent)
print("Tree spatial reference:", arcpy.Describe(tree_fc).spatialReference.name)

In [ ]:
STREETS = str(GDB/"streets")
ARTERIAL_STREETS = str(GDB/ "arterial_streets")

major_street_classes = [
    "Principal Arterial",
    "Minor Arterial",
    "Collector Arterial"
]

field = arcpy.AddFieldDelimiters(STREETS, "ARTDESCRIPT")

values = ", ".join(
    f"'{value.replace(chr(39), chr(39) *2)}'"
    for value in major_street_classes
)

where_clause = f"{field} IN ({values})"

if arcpy.Exists(ARTERIAL_STREETS):
    arcpy.management.Delete(ARTERIAL_STREETS)

arcpy.analysis.Select(
    in_features=STREETS,
    out_feature_class=ARTERIAL_STREETS,
    where_clause=where_clause,
)

street_count = int(arcpy.management.GetCount(STREETS)[0])
arterial_count = int(arcpy.management.GetCount(ARTERIAL_STREETS)[0])

print(f"All street segments: {street_count:,}")
print(f"Selected arterial segments: {arterial_count:,}")
print(f"Output: {ARTERIAL_STREETS}")